# Yambda CF positive_pairs + item_frequencies

Аналог [tiger/cf_dataset_builder_vk.ipynb](cf_dataset_builder_vk.ipynb). Из `inter.json` строим скользящим окном пары (anchor, positive), отбрасывая 2 последних элемента истории (для leave-one-out внутри TIGER), и считаем частоты айтемов по `history[:-2]` для logQ-варианта.

In [1]:
from collections import defaultdict
from typing import Iterator, Tuple

import json
import numpy as np
import pickle

In [2]:
base_dir = '../data/yambda/'
embeddings_input_path = base_dir + 'content_embeddings.pkl'
inter_json_path = base_dir + 'inter.json'
positive_pairs_path = base_dir + 'positive_pairs.txt'
item_frequencies_path = base_dir + 'item_frequencies.txt'

## Загрузка эмбедов (для sanity-чека размеров)

In [3]:
with open(embeddings_input_path, 'rb') as f:
    data = pickle.load(f)

item_ids = np.array(data['item_id'], dtype=np.int64)
X = np.array(data['embedding'], dtype=np.float32)
X.shape

(33029, 128)

## Загрузка interactions

In [4]:
with open(inter_json_path, 'r') as f:
    user_interactions = json.load(f)

max_item_id = 0
for user_id_str, items in user_interactions.items():
    user_id = int(user_id_str)
    if items:
        max_item_id = max(max_item_id, max(items))
    assert len(items) >= 5, f'Core-5 dataset is used, user {user_id} has only {len(items)} items'
max_item_id

33028

## Подсчёт частот айтемов

Частоты считаем по `history[:-2]` — последние 2 элемента (val/test) исключаем, чтобы не «подсматривать» в таргет. Используется в logQ-варианте CF-finetune (см. `tiger/cf_finetune_log_q_yambda.ipynb`).

In [5]:
item_frequency_counts = {}
for user_id_str, history in user_interactions.items():
    usable_items = history[:-2] if len(history) > 2 else []
    for item_id in usable_items:
        item_frequency_counts[item_id] = item_frequency_counts.get(item_id, 0) + 1

num_items = max_item_id + 1
item_frequencies = [0] * num_items
for item_id, count in item_frequency_counts.items():
    if item_id < 0 or item_id >= num_items:
        raise ValueError(f'Invalid item ID: {item_id}')
    item_frequencies[item_id] = count

with open(item_frequencies_path, 'w', encoding='utf-8') as f:
    for count in item_frequencies:
        f.write(f'{count}\n')

print(f'item_frequencies.txt сохранён: {item_frequencies_path} ({num_items} строк)')

item_frequencies.txt сохранён: ../data/yambda/item_frequencies.txt (33029 строк)


## Построение пар

`drop_last_cnt=2` — последние 2 айтема истории резервируются под val/test внутри TIGER (см. `modeling/dataset/base.py`).

In [6]:
def calc_pairs(history, drop_last_cnt=2) -> Iterator[Tuple[int, int]]:
    """
    in the modeling/dataset we drop last elements for leave one out
    e.g. sequence = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] (leave one out scheme, 8 - train, 9 - valid, 10 - test)
    so I decided to drop 2 last item_ids too
    """
    window_size = 2
    start_till = len(history) - window_size + 1 - drop_last_cnt
    start_till = max(0, start_till)
    for start in range(start_till):
        window = history[start : start + window_size]
        yield window[0], window[-1]

In [7]:
def build_dataset(user_interactions):
    positive_pairs = []
    for history in user_interactions.values():
        positive_pairs.extend(calc_pairs(history))
    return positive_pairs

In [8]:
pairs = build_dataset(user_interactions)
len(pairs)

611328

## Sanity: нет self-pairs

In [9]:
cnt = 0
for (a, b) in pairs:
    if a == b:
        cnt += 1
cnt

7700

## Сохранение

In [10]:
with open(positive_pairs_path, 'w', encoding='utf-8') as f:
    for first, second in pairs:
        f.write(f'{first} {second}\n')

print(f'positive_pairs.txt сохранён: {positive_pairs_path} ({len(pairs)} строк)')

positive_pairs.txt сохранён: ../data/yambda/positive_pairs.txt (611328 строк)
